# Profiling and Optimizing the Image Mosaic Generator

## By - Joaquin Saldarriaga (NUID: 002597882)

## CS5130 - Northeastern University

## Objectives

In this activity, we take the '_Image Mosaic Generator_' of **Lab 1** and improve it in three steps - _first make it work_ (Lab 1), _then make it fast_, _then make it maintainable_:

→ **Part 1 (profiling):** Time my Lab 1 code on **images of three sizes** and **three grid sizes**, then find the bottlenecks with '```cProfile```' (function level) and '```line_profiler```' (line level)

→ **Part 2 (optimization):** Remove the bottlenecks one at a time (vectorization, a k-d tree, caching, a single resize), measure the **speedup of every step**, and check that the output did **not** change

→ **Part 3 (refactoring):** Show the modular '```mosaic_generator```' package that contains the optimized code, with its API and unit tests

All numbers in this Notebook are produced by the cells below, on the machine listed in the next section.

## Part 1: Profiling the Lab 1 Code

### Setup and baseline

 * **Baseline used:** my **own Lab 1 code** (the folder '```baseline_lab1```', copied from the Lab 1 submission) - _not_ the reference implementation
    * Lab 1 **works** (it produced the preprocessed, segmented and mosaic images for my sample photos) and, as the profile below shows, it is **not** already well optimized
 * **One change only:** '```create_mosaic```' gets an extra '```image_size```' argument (default $512$, the value that Lab 1 hard-coded), so that the three required image sizes can be tested - the algorithm is untouched
 * The libraries '```cProfile```', '```pstats```' and '```line_profiler```' are used for the analysis, '```numpy```' and '```pandas```' for the data, and '```matplotlib```' for the figures

In [ ]:
## ********** Import Required Libraries **********
import contextlib
import cProfile
import io
import json
import os
import platform
import pstats
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import skimage
from PIL import Image
from scipy.spatial import cKDTree
from skimage.metrics import structural_similarity

## Project folder (this Notebook lives in it) and the folder with the Lab 1 baseline code
PROJECT = Path.cwd()

## Folder of the results
RESULTS = PROJECT / "results"

## Create the results folder if needed
RESULTS.mkdir(exist_ok = True)

## Make both code bases importable
sys.path.insert(0, str(PROJECT))

## Make the Lab 1 baseline importable
sys.path.insert(0, str(PROJECT / "baseline_lab1"))

## Lab 1 baseline modules (flat imports, exactly as submitted in Lab 1)
import benchmarking
import grid_analysis
import metrics as lab1_metrics
import mosaic_mapping
import mosaic_pipeline
import preprocessing
import tile_library

## Lab 8 optimized package (Part 3 explains its structure)
from mosaic_generator import MosaicBuilder, TileManager
from mosaic_generator import image_processor
from mosaic_generator import metrics as lab8_metrics
from mosaic_generator.utils import format_seconds, time_callable

## Cleaner table printing
pd.options.display.float_format = "{:,.4f}".format

## Use a wide display for the tables
pd.options.display.width = 200

In [ ]:
## ********** Machine used for every timing below (timings depend on it!) **********
machine = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "scikit-image": skimage.__version__,
    "cpu_count": os.cpu_count(),
    "platform": platform.platform(),
}

## Show every entry on its own line
for key, value in machine.items():
    ## Print every item of the environment
    print(f"{key:<13}:", value)

In [ ]:
## ********** Figure style: thin marks, quiet grid, one color per version (blue = baseline, orange = optimized) **********
COLORS = {"baseline": "#2a78d6", "optimized": "#eb6834", "third": "#1baf7a"}

## Primary text color
INK = "#0b0b0b"

## Secondary text color
INK_SECONDARY = "#52514e"

## Quiet grid color
GRID = "#e5e4e0"

## Background color of the figures
SURFACE = "#fcfcfb"

## Apply the style to every figure of the Notebook
plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": INK_SECONDARY,
    "axes.labelcolor": INK,
    "axes.titlecolor": INK,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "xtick.color": INK_SECONDARY,
    "ytick.color": INK_SECONDARY,
    "text.color": INK,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "legend.frameon": False,
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "savefig.facecolor": SURFACE,
})

### Test suite

 * **Images:** one photo (a center-cropped square of my first sample image) resized **once** to '$256\times256$', '$512\times512$' and '$1024\times1024$'
 * **Grids:** '$16\times16$', '$32\times32$' and '$64\times64$' cells - all **nine** combinations are timed (the three _required_ ones are '$256$ / $16$', '$512$ / $32$' and '$1024$ / $64$')
 * **Tiles:** the Lab 1 '```Diagonal```' family (four procedural tiles), the default of the Lab 1 interface
 * **Repetitions:** the baseline runs **5 times** per configuration after one warm-up call and the optimized code **30 times** (it is much faster); the **mean** is reported together with the standard deviation

In [ ]:
## ********** Test Suite: square images of three sizes **********
SAMPLE = PROJECT / "sample_images" / "sample_image_joaquin_1.jpeg"

## Image sides to test
IMAGE_SIZES = (256, 512, 1024)

## Grid sizes to test
GRID_SIZES = (16, 32, 64)

## Tile family used in every test
TILE_SET = "Diagonal"

## Repetitions of the slow Lab 1 code
BASELINE_RUNS = 5

## Repetitions of the fast Lab 8 code
OPTIMIZED_RUNS = 30

## Center-crop the sample photo to a square
photo = Image.open(SAMPLE).convert("RGB")

## Side of the largest centered square
side = min(photo.size)

## Left and top borders of the square
left, top = (photo.width - side) // 2, (photo.height - side) // 2

## Crop the centered square
square = photo.crop((left, top, left + side, top + side))

## Resize the square once to every test size
test_images = {size: np.asarray(square.resize((size, size), Image.Resampling.LANCZOS)) for size in IMAGE_SIZES}

## Show the three test images
fig, axes = plt.subplots(1, 3, figsize = (10, 3.6))

## Loop over the test images
for axis, (size, image) in zip(axes, test_images.items()):
    ## Show the image
    axis.imshow(image)

    ## Title with the image size
    axis.set_title(f"{size} x {size}")

    ## Hide the axes
    axis.axis("off")

## Avoid overlapping elements
plt.tight_layout()

## Display the figure
plt.show()

### Baseline performance of Lab 1

 * '```create_mosaic```' of Lab 1 is timed **as submitted**: it preprocesses the image, builds the segmentation and the mosaic, computes the metrics (MSE, SSIM) and runs the loop-vs-vectorized benchmark - the report it prints is hidden here only to keep the Notebook readable
 * Wall-clock time with '```time.perf_counter```' (through the helper '```time_callable```' of the package)

In [ ]:
## ********** Baseline timing: all nine image-size / grid-size combinations **********
## This function runs the Lab 1 pipeline once and hides the report it prints
def run_baseline(image: np.ndarray, grid: int, size: int):
    """ Call the Lab 1 create_mosaic function with its printed report hidden """

    ## Redirect the prints of Lab 1 to a throw-away buffer
    with contextlib.redirect_stdout(io.StringIO()):
        ## Run the full Lab 1 pipeline
        return mosaic_pipeline.create_mosaic(image, grid, TILE_SET, size)


## One row per combination
baseline_rows = []

## Loop over the image sizes
for size in IMAGE_SIZES:
    ## Loop over the grid sizes
    for grid in GRID_SIZES:

        ## Time the Lab 1 pipeline (warm-up call first, then the timed runs)
        summary = time_callable(lambda: run_baseline(test_images[size], grid, size), repeats = BASELINE_RUNS, warmup = 1)

        ## Keep the statistics of the runs
        baseline_rows.append({
            "Image size": f"{size}x{size}", "Grid size": f"{grid}x{grid}", "Cell (px)": size // grid,
            "Mean (s)": summary["mean"], "Std (s)": summary["std"], "Min (s)": summary["min"],
            "size": size, "grid": grid,
        })

## Show the table without the helper columns
baseline_table = pd.DataFrame(baseline_rows)

## Display the table without the helper columns
baseline_table.drop(columns = ["size", "grid"])

### Which operations take the most time?

 * First look at the **stages** of one Lab 1 call at the middle configuration ('$512\times512$', '$32\times32$'): every Lab 1 function is timed separately (mean of $10$ calls)
 * The two parts of the metrics ('```MSE```' and '```SSIM```') are separated, because they behave very differently

In [ ]:
## ********** Stage-by-stage timing of the Lab 1 pipeline at 512 x 512 / 32 x 32 **********
## This function times one stage (mean seconds over several calls)
def stage_time(func, repeats: int = 10) -> float:
    """ Mean wall-clock seconds of func() over several calls (one warm-up call first) """

    ## Return only the mean of the summary
    return time_callable(func, repeats = repeats, warmup = 1)["mean"]


## Inputs of every stage, produced once
image_512, grid_512 = test_images[512], 32

## Preprocessing stage
prepared_512 = preprocessing.prepare_image(image_512, (512, 512))

## Grid analysis stage
means_512 = grid_analysis.cell_means_vectorized(prepared_512, grid_512)

## Tile library stage
tiles_512 = tile_library.create_tile_set(512 // grid_512, TILE_SET)

## Mosaic reconstruction stage
mosaic_512 = mosaic_mapping.reconstruct_mosaic(means_512, tiles_512)

## One entry per stage of create_mosaic, in the order in which they run
baseline_stages = {
    "prepare_image (crop + LANCZOS resize)": stage_time(lambda: preprocessing.prepare_image(image_512, (512, 512))),
    "cell_means_vectorized": stage_time(lambda: grid_analysis.cell_means_vectorized(prepared_512, grid_512)),
    "segmented_image": stage_time(lambda: grid_analysis.segmented_image(prepared_512, means_512)),
    "create_tile_set (rebuilt on every call)": stage_time(lambda: tile_library.create_tile_set(512 // grid_512, TILE_SET)),
    "reconstruct_mosaic": stage_time(lambda: mosaic_mapping.reconstruct_mosaic(means_512, tiles_512)),
    "MSE": stage_time(lambda: lab1_metrics.mean_squared_error(prepared_512, mosaic_512)),
    "SSIM": stage_time(lambda: lab1_metrics.structural_similarity_index(prepared_512, mosaic_512), repeats = 5),
    "benchmark_grid_operations (loops vs. vectorized)": stage_time(lambda: benchmarking.benchmark_grid_operations(prepared_512), repeats = 3),
}

## Share of every stage in the sum of the stages
stage_table = pd.DataFrame({"Time (ms)": {name: seconds * 1e3 for name, seconds in baseline_stages.items()}})

## Share of every stage in the total
stage_table["Share (%)"] = 100 * stage_table["Time (ms)"] / stage_table["Time (ms)"].sum()

## Display the table
stage_table

In [ ]:
## ********** Figure: share of every stage of the Lab 1 pipeline **********
## Sort the stages so that the biggest one is on top
ordered = stage_table.sort_values("Time (ms)")

## Create the figure
fig, ax = plt.subplots(figsize = (8.6, 3.8))

## Horizontal bars in one color, labelled with their time
bars = ax.barh(ordered.index, ordered["Time (ms)"], color = COLORS["baseline"], height = 0.6)

## Loop over the bars
for bar, (_, row) in zip(bars, ordered.iterrows()):
    ## Label every bar with its time and share
    ax.text(bar.get_width() + 0.8, bar.get_y() + bar.get_height() / 2, f"{row['Time (ms)']:.1f} ms ({row['Share (%)']:.0f} %)", va = "center", fontsize = 9, color = INK_SECONDARY)

## Axes, title and limits
ax.set_xlabel("Time of one call (ms)")

## Title of the figure
ax.set_title("Where the time goes in Lab 1 (512 x 512 image, 32 x 32 grid)", loc = "left")

## Leave room for the labels
ax.set_xlim(0, ordered["Time (ms)"].max() * 1.35)

## Hide the horizontal grid
ax.grid(axis = "y", visible = False)

## Avoid overlapping elements
plt.tight_layout()

## Save the figure
plt.savefig(RESULTS / "fig_stage_breakdown.png")

## Display the figure
plt.show()

### Function-level analysis with cProfile

 * '```cProfile```' is a _deterministic_ profiler: it records **every function call** ('```ncalls```'), the time **inside** each function ('```tottime```') and the time **including** the functions it calls ('```cumtime```')
 * We profile the **main mosaic generation function** ('```create_mosaic```') on the middle configuration, over $10$ calls so that the numbers are stable
 * First sorted by **cumulative time** (_who is responsible_), then by **total time** (_where the CPU really spends it_)

In [ ]:
## ********** cProfile: Lab 1 create_mosaic, 10 calls at 512 x 512 / 32 x 32 **********
profiler = cProfile.Profile()

## Start the profiler
profiler.enable()

## Run the baseline ten times
for _ in range(10):
    ## One baseline call
    run_baseline(test_images[512], 32, 512)

## Stop the profiler
profiler.disable()

## Report sorted by cumulative time (top 15) - kept in a buffer so that it can also be saved for the report
buffer = io.StringIO()

## Sort by cumulative time and print the top 15
pstats.Stats(profiler, stream = buffer).strip_dirs().sort_stats("cumulative").print_stats(15)

## Keep the text of the report
cprofile_cumulative = buffer.getvalue()

## Show the report
print(cprofile_cumulative)

In [ ]:
## ********** cProfile: the same 10 calls sorted by total time (top 8) **********
buffer = io.StringIO()

## Sort by own time and print the top 8
pstats.Stats(profiler, stream = buffer).strip_dirs().sort_stats("tottime").print_stats(8)

## Keep the text of the report
cprofile_tottime = buffer.getvalue()

## Show the report
print(cprofile_tottime)

## Save both reports for the performance report
(RESULTS / "cprofile_baseline.txt").write_text("== sorted by cumulative time ==\n" + cprofile_cumulative + "\n== sorted by total time ==\n" + cprofile_tottime, encoding = "utf-8")

**What the function-level profile shows:**
 * '```create_mosaic```' is the root of the report - everything else is a callee, and the **two largest branches** are '```compare_images```' (the _SSIM_ of '```scikit-image```', which calls '```scipy```' uniform filters several times) and '```benchmark_grid_operations```' (which calls '```cell_means_loop```' for every cell)
 * The call counts expose an **unexpected bottleneck**: '```cell_means_loop```' and the '```ndarray.mean```' calls inside it run **once per grid cell** at three grid sizes ($256+1{,}024+4{,}096$ cells per call) - a benchmark that **Lab 1 executes inside every mosaic request**
 * The mosaic work itself ('```cell_means_vectorized```', '```segmented_image```', '```reconstruct_mosaic```') is already vectorized and appears _far_ down the list
 * No function reads files, so there is **no repeated file I/O** - but '```create_tile_set```' is called on _every_ request, although the tiles never change

### Line-level analysis with line_profiler

 * '```cProfile```' stops at the function boundary, so '```line_profiler```' (loaded with '```%load_ext line_profiler```') is used to see **which lines** of the slow functions cost the time
 * '```%lprun -r```' returns the profile object, whose report is printed in **milliseconds**
 * First '```create_mosaic```' and the two functions that dominate its cProfile: '```benchmark_grid_operations```' and '```cell_means_loop```'

In [ ]:
## ********** Load the line_profiler extension **********
%load_ext line_profiler

In [ ]:
## ********** Helper: keep only the lines of the report that were executed **********
import re

## This function removes docstring and blank lines (no hits) from a line_profiler report
def executed_lines(report: str) -> str:
    """ Keep the report header and the lines that have a hit count, a time and a percentage """

    ## A line of source code that ran has: line number, hits, time, per hit and % time
    executed = re.compile(r"^\s*\d+\s+\d+\s+[\d.]+\s+[\d.]+\s+[\d.]+")

    ## Keep the header lines (timer unit, total time, file, function, column titles) and the executed lines
    kept = [line for line in report.splitlines() if executed.match(line) or not re.match(r"^\s*\d+\s", line)]

    ## Return the report without empty lines
    return "\n".join(line for line in kept if line.strip())

In [ ]:
## ********** line_profiler: create_mosaic, the benchmark and the loop-based cell means **********
lp_pipeline = %lprun -r -f mosaic_pipeline.create_mosaic -f benchmarking.benchmark_grid_operations -f grid_analysis.cell_means_loop run_baseline(test_images[512], 32, 512)

## Print the report in milliseconds and keep a copy for the performance report
buffer = io.StringIO()

## Print the statistics in milliseconds
lp_pipeline.print_stats(stream = buffer, output_unit = 1e-3, stripzeros = True)

## Keep only the executed lines
lprof_pipeline = executed_lines(buffer.getvalue())

## Show the report
print(lprof_pipeline)

 * Next the metrics: '```compare_images```' (Lab 1) and the '```structural_similarity```' function of '```scikit-image```' that it calls

In [ ]:
## ********** line_profiler: the metrics of Lab 1 and the SSIM routine they call **********
lp_metrics = %lprun -r -f lab1_metrics.compare_images -f structural_similarity lab1_metrics.compare_images(prepared_512, mosaic_512)

## Print the report in milliseconds and keep a copy
buffer = io.StringIO()

## Print the statistics in milliseconds
lp_metrics.print_stats(stream = buffer, output_unit = 1e-3, stripzeros = True)

## Keep only the executed lines
lprof_metrics = executed_lines(buffer.getvalue())

## Show the report
print(lprof_metrics)

In [ ]:
## ********** line_profiler: the vectorized grid stages of Lab 1 **********
## This function runs the four vectorized stages of Lab 1 once
def run_grid_stages():
    """ Cell means, segmentation, tile set and mosaic reconstruction of Lab 1 at 512 x 512 / 32 x 32 """

    ## Cell colors and the block-segmented picture
    means = grid_analysis.cell_means_vectorized(prepared_512, 32)

    ## Segmented picture
    grid_analysis.segmented_image(prepared_512, means)

    ## Tile set and mosaic
    tiles = tile_library.create_tile_set(16, TILE_SET)

    ## Mosaic from the means
    mosaic_mapping.reconstruct_mosaic(means, tiles)


## Profile the four functions line by line
lp_grid = %lprun -r -f grid_analysis.cell_means_vectorized -f grid_analysis.segmented_image -f mosaic_mapping.reconstruct_mosaic -f tile_library.create_tile_set run_grid_stages()

## Print the report in milliseconds and keep a copy
buffer = io.StringIO()

## Print the statistics in milliseconds
lp_grid.print_stats(stream = buffer, output_unit = 1e-3, stripzeros = True)

## Keep only the executed lines
lprof_grid = executed_lines(buffer.getvalue())

## Show the report
print(lprof_grid)

## Save the three line-level reports for the performance report
(RESULTS / "lineprofiler_baseline.txt").write_text(lprof_pipeline + "\n" + lprof_metrics + "\n" + lprof_grid, encoding = "utf-8")

### Bottlenecks identified

From the three views above (stage times, cProfile, line_profiler) the following **specific bottlenecks** are targeted - the table below computes the share of each one from the stage times measured earlier:

 * **Bottleneck 1 - the loop benchmark runs inside every request:** '```benchmark_grid_operations```' calls '```cell_means_loop```' (a double Python loop with one '```ndarray.mean```' per cell) and the vectorized version at three grid sizes; line_profiler shows the lines of the loop that slice the cell and call '```mean```' dominating
 * **Bottleneck 2 - the similarity metrics run inside every request:** the _SSIM_ is a sliding-window statistic (seven filters over the full RGB image, see the lines of '```structural_similarity```') and is computed even when nobody asks for it
 * **Bottleneck 3 - the cell reduction itself:** '```cell_means_vectorized```' transposes the image into a **non-contiguous** view and then averages it in '```float32```' - NumPy must walk memory with large strides
 * **Bottleneck 4 - recomputed constants:** the tile set (and its mean colors) is rebuilt on every call, and the segmentation image is built even when only the mosaic is needed
 * **Bottleneck 5 - preprocessing of large photos:** '```prepare_image```' converts the array to a PIL image and runs a '```LANCZOS```' resize on the full-size photo (cheap for the test images, dominant for a camera photo)

In [ ]:
## ********** Summary of the bottlenecks with their measured share **********
## Shares of the stage table (percent of the Lab 1 call at 512 x 512 / 32 x 32)
share = stage_table["Share (%)"]

## One row per bottleneck, with the evidence and the planned fix
bottleneck_table = pd.DataFrame([
    ["1. Loop benchmark inside create_mosaic", "benchmark_grid_operations -> cell_means_loop", share["benchmark_grid_operations (loops vs. vectorized)"], "Remove it from the request path (keep it as a separate tool)"],
    ["2. Metrics inside create_mosaic", "compare_images -> structural_similarity", share["SSIM"] + share["MSE"], "Compute on demand: compute_similarity()"],
    ["3. Strided float32 cell reduction", "cell_means_vectorized", share["cell_means_vectorized"], "Two contiguous integer reductions (uint16, uint32)"],
    ["4. Tile set and segmentation rebuilt", "create_tile_set, segmented_image", share["create_tile_set (rebuilt on every call)"] + share["segmented_image"], "Cache tiles and features; build the segmentation only when asked"],
    ["5. Resize of large photos", "prepare_image", share["prepare_image (crop + LANCZOS resize)"], "One crop + one resize; optional cheaper filter"],
], columns = ["Bottleneck", "Where", "Share of the call (%)", "Fix"])

## Display the table of bottlenecks
bottleneck_table

## Part 2: Optimizing for Performance

### Optimization strategy

Every optimization targets **one** bottleneck of the previous section, and is measured **on its own** before the next one is added (_one change at a time_, _verify after each one_):

 * **Step 1 (Bottleneck 1):** the loop-vs-vectorized benchmark leaves the request path - it is a measurement _tool_, not part of building a mosaic
 * **Step 2 (Bottleneck 2):** the metrics are computed **on demand** ('```compute_similarity```'), not in every request
 * **Step 3 (Bottleneck 3):** cell means become **two contiguous integer reductions** (rows first, then columns; '```uint16```' and '```uint32```' accumulators are exact) and the segmentation is **broadcast** instead of two '```np.repeat```' calls
 * **Step 4 (Bottleneck 4):** tiles, their mean colors (and a '```cKDTree```' for big libraries) are computed **once per cell size** and cached; **all** cells are matched with **one** vectorized nearest-neighbor query - a matrix product for tiny tile sets, the k-d tree for large ones (chosen automatically, see the library experiment)
 * **Step 5:** the segmentation image is built **only when requested**
 * **Resizing:** the input is cropped and resized **once** ('```prepare_image```'), and the resampling filter is selectable (studied at the end)

### The optimization ladder at '$512\times512$' / '$32\times32$'

 * '```V0```' is Lab 1 **as submitted**; every next version **adds** one optimization and keeps the previous ones
 * Besides the time, every version is checked for **correctness**: its mosaic must be **exactly equal** to the Lab 1 mosaic (pixel by pixel)
 * '```V1```' - '```V3```' are written with the Lab 1 functions plus the optimized ones, so that each step can be isolated; '```V4```' and '```V5```' are the final package ('```MosaicBuilder```')
 * '```V4```' keeps the segmentation image, so that it returns the **same three images** as Lab 1; '```V5```' returns only the mosaic

In [ ]:
## ********** Ladder inputs: 512 x 512 test image, 32 x 32 grid, Diagonal tiles **********
LADDER_IMAGE, LADDER_GRID, LADDER_SIZE = test_images[512], 32, 512

## This function is Version 1: Lab 1 without the loop benchmark
def pipeline_v1(image: np.ndarray, grid: int, size: int):
    """ Lab 1 pipeline with the loop-vs-vectorized benchmark removed (metrics still computed) """

    ## The same Lab 1 stages, in the same order
    prepared = preprocessing.prepare_image(image, (size, size))

    ## Mean color of every cell
    means = grid_analysis.cell_means_vectorized(prepared, grid)

    ## Segmented image
    segmented = grid_analysis.segmented_image(prepared, means)

    ## Tile library
    tiles = tile_library.create_tile_set(size // grid, TILE_SET)

    ## Mosaic from the means
    mosaic = mosaic_mapping.reconstruct_mosaic(means, tiles)

    ## The metrics are still computed here
    lab1_metrics.compare_images(prepared, mosaic)

    ## Return the three images
    return prepared, segmented, mosaic


## This function is Version 2: metrics on demand
def pipeline_v2(image: np.ndarray, grid: int, size: int):
    """ Lab 1 pipeline without benchmark and without metrics (they are computed only when asked) """

    ## The same Lab 1 stages as Version 1, without the metrics
    prepared = preprocessing.prepare_image(image, (size, size))

    ## Mean color of every cell
    means = grid_analysis.cell_means_vectorized(prepared, grid)

    ## Segmented image
    segmented = grid_analysis.segmented_image(prepared, means)

    ## Tile library
    tiles = tile_library.create_tile_set(size // grid, TILE_SET)

    ## Mosaic from the means
    mosaic = mosaic_mapping.reconstruct_mosaic(means, tiles)

    ## Return the three images
    return prepared, segmented, mosaic


## This function is Version 3: contiguous integer cell reduction and broadcast segmentation
def pipeline_v3(image: np.ndarray, grid: int, size: int):
    """ Version 2 with the optimized cell means and segmentation (tiles are still rebuilt every call) """

    ## Preprocessing and tiles stay as in Lab 1
    prepared = preprocessing.prepare_image(image, (size, size))

    ## Cell size
    cell = size // grid

    ## Optimized cell means and segmentation
    means = image_processor.cell_means(prepared, grid)

    ## Segmented image with the new function
    segmented = image_processor.segmented_image(means, cell, cell)

    ## Tile creation and matching stay as in Lab 1
    tiles = tile_library.create_tile_set(cell, TILE_SET)

    ## Mosaic from the means
    mosaic = mosaic_mapping.reconstruct_mosaic(means, tiles)

    ## Return the three images
    return prepared, segmented, mosaic


## Final package: one builder with cached tiles, k-d tree matching and lazy segmentation
tile_manager = TileManager(tile_set = TILE_SET)

## Builder with the cached tiles
builder_512 = MosaicBuilder(tile_manager, grid_size = LADDER_GRID, image_size = LADDER_SIZE)


## This function is Version 4: the package with the segmentation image
def pipeline_v4(image: np.ndarray, grid: int, size: int):
    """ MosaicBuilder.build with the segmentation (the same three images as Lab 1) """

    ## One call of the final package
    result = builder_512.build(image, segmentation = True)

    ## Return the three images
    return result.prepared, result.segmented, result.mosaic


## This function is Version 5: the package without the segmentation image
def pipeline_v5(image: np.ndarray, grid: int, size: int):
    """ MosaicBuilder.create_mosaic: only the mosaic, which is what most callers need """

    ## Only the mosaic is built
    return None, None, builder_512.create_mosaic(image)

In [ ]:
## ********** Measure the ladder: time, speedup of the step, cumulative speedup, correctness **********
ladder_versions = [
    ("V0  Lab 1 as submitted", lambda: run_baseline(LADDER_IMAGE, LADDER_GRID, LADDER_SIZE), BASELINE_RUNS),
    ("V1  + loop benchmark removed from the request", lambda: pipeline_v1(LADDER_IMAGE, LADDER_GRID, LADDER_SIZE), 10),
    ("V2  + metrics on demand", lambda: pipeline_v2(LADDER_IMAGE, LADDER_GRID, LADDER_SIZE), 20),
    ("V3  + integer cell reduction, broadcast segmentation", lambda: pipeline_v3(LADDER_IMAGE, LADDER_GRID, LADDER_SIZE), 30),
    ("V4  + cached tiles, one vectorized match (MosaicBuilder.build)", lambda: pipeline_v4(LADDER_IMAGE, LADDER_GRID, LADDER_SIZE), 30),
    ("V5  + lazy segmentation (MosaicBuilder.create_mosaic)", lambda: pipeline_v5(LADDER_IMAGE, LADDER_GRID, LADDER_SIZE), 30),
]

## Reference mosaic of Lab 1 for the correctness column
reference_mosaic = run_baseline(LADDER_IMAGE, LADDER_GRID, LADDER_SIZE)[2]

## Time every version and check that its mosaic equals the Lab 1 mosaic
ladder_rows = []

## Loop over the versions
for name, call, runs in ladder_versions:
    ## Time the version
    summary = time_callable(call, repeats = runs, warmup = 2)

    ## Keep the mosaic of the version
    mosaic = call()[2]

    ## Store the row of the table
    ladder_rows.append({"Version": name, "Mean (ms)": summary["mean"] * 1e3, "Std (ms)": summary["std"] * 1e3, "Identical to Lab 1 mosaic": bool(np.array_equal(mosaic, reference_mosaic))})

## Speedups: against the previous step and against Lab 1
ladder_table = pd.DataFrame(ladder_rows).set_index("Version")

## Speedup of every step
ladder_table["Speedup of this step"] = ladder_table["Mean (ms)"].shift(1) / ladder_table["Mean (ms)"]

## Cumulative speedup
ladder_table["Cumulative speedup"] = ladder_table["Mean (ms)"].iloc[0] / ladder_table["Mean (ms)"]

## Display the table
ladder_table

In [ ]:
## ********** Figure: time after every optimization step **********
fig, ax = plt.subplots(figsize = (9, 3.8))

## Short labels of the versions
labels = [name.split("  ")[0] for name in ladder_table.index]

## Times in milliseconds
values = ladder_table["Mean (ms)"].to_numpy()

## One bar per version, labelled with the time and the cumulative speedup
bars = ax.bar(labels, values, color = COLORS["baseline"], width = 0.55)

## Highlight the final version
bars[-1].set_color(COLORS["optimized"])

## Loop over the bars
for bar, value, speedup in zip(bars, values, ladder_table["Cumulative speedup"]):
    ## Label every bar with its time and speedup
    ax.text(bar.get_x() + bar.get_width() / 2, value * 1.15, f"{value:.1f} ms\n{speedup:.0f}x", ha = "center", fontsize = 9, color = INK_SECONDARY)

## Log axis because the steps differ by orders of magnitude
ax.set_yscale("log")

## Room for the labels
ax.set_ylim(values.min() / 2, values.max() * 6)

## Label of the vertical axis
ax.set_ylabel("Time of one call (ms, log scale)")

## Title of the figure
ax.set_title("Optimization ladder (512 x 512 image, 32 x 32 grid)", loc = "left")

## Hide the vertical grid
ax.grid(axis = "x", visible = False)

## Avoid overlapping elements
plt.tight_layout()

## Save the figure
plt.savefig(RESULTS / "fig_ladder.png")

## Display the figure
plt.show()

**Reading the ladder:**
 * The **first two steps** remove most of the time because Lab 1 spent most of every request on **two tasks that are not mosaic construction** (the benchmark and the SSIM) - the profiler found them, a micro-optimization of the mosaic code alone would have missed them
 * The step that rewrites the **cell reduction** and the one that **caches** the tiles are small in absolute terms, but they decide how far the time can still fall once the big items are gone (the remaining time is the sum of a few _millisecond-sized_ stages)
 * **Every** version keeps the mosaic **bit-identical** to Lab 1 - the optimizations change _how_ the result is computed, not _what_ is computed

### Before / after on the nine configurations

Three comparisons are reported, from the one that matches the assignment ('_Lab 8 time_') to the most conservative:

 * **Lab 8 time:** '```MosaicBuilder.build```' returns the **same three images as Lab 1** (preprocessed, segmented, mosaic); the metrics are computed on demand ('```compute_similarity```') and the benchmark is not part of a request
 * **Lab 8 + metrics:** the same call **plus** MSE, PSNR and SSIM, which is the closest equivalent of what Lab 1 did in every request (still without the benchmark, which is not mosaic construction)
 * **Core only:** Lab 1 **without** metrics and benchmark (Version 2 above) against '```build```' - the speedup that comes **only** from the faster mosaic code

In [ ]:
## ********** Optimized timing: all nine image-size / grid-size combinations **********
optimized_rows = []

## Loop over the image sizes
for size in IMAGE_SIZES:
    ## Loop over the grid sizes
    for grid in GRID_SIZES:
        ## Image of this size
        image = test_images[size]

        ## A fresh builder and tile manager, so that the first call is a true cold start
        builder = MosaicBuilder(TileManager(tile_set = TILE_SET), grid_size = grid, image_size = size)

        ## Start the clock
        start = time.perf_counter()

        ## First build: includes the cache fill
        builder.build(image)

        ## Cold time
        cold = time.perf_counter() - start

        ## Warm calls: the same three images as Lab 1, then with the metrics, then Lab 1 without metrics and benchmark
        warm = time_callable(lambda: builder.build(image, segmentation = True), repeats = OPTIMIZED_RUNS, warmup = 2)

        ## Time with the metrics
        with_metrics = time_callable(lambda: builder.compute_similarity(image, builder.build(image).mosaic), repeats = 10, warmup = 1)

        ## Time of the core of Lab 1
        core_baseline = time_callable(lambda: pipeline_v2(image, grid, size), repeats = 10, warmup = 1)

        ## Time of the mosaic only
        mosaic_only = time_callable(lambda: builder.create_mosaic(image), repeats = OPTIMIZED_RUNS, warmup = 2)

        ## Keep every measurement
        optimized_rows.append({
            "size": size, "grid": grid, "cold": cold,
            "lab8": warm["mean"], "lab8_std": warm["std"],
            "lab8_metrics": with_metrics["mean"], "core_baseline": core_baseline["mean"],
            "mosaic_only": mosaic_only["mean"],
        })

## Combine with the baseline table
comparison = baseline_table.merge(pd.DataFrame(optimized_rows), on = ["size", "grid"])

## Speedup of the full request
comparison["Speedup"] = comparison["Mean (s)"] / comparison["lab8"]

## Speedup including the metrics
comparison["Speedup (+ metrics)"] = comparison["Mean (s)"] / comparison["lab8_metrics"]

## Speedup of the core only
comparison["Speedup (core only)"] = comparison["core_baseline"] / comparison["lab8"]

## Readable table for the nine configurations
comparison_table = comparison[["Image size", "Grid size", "Mean (s)", "lab8", "Speedup", "lab8_metrics", "Speedup (+ metrics)", "core_baseline", "Speedup (core only)"]].rename(columns = {
    "Mean (s)": "Baseline (s)", "lab8": "Lab 8 (s)", "lab8_metrics": "Lab 8 + metrics (s)", "core_baseline": "Lab 1 core (s)",
})

## Display the table
comparison_table.style.format({"Baseline (s)": "{:.4f}", "Lab 8 (s)": "{:.5f}", "Speedup": "{:,.1f}x", "Lab 8 + metrics (s)": "{:.4f}",
                               "Speedup (+ metrics)": "{:,.1f}x", "Lab 1 core (s)": "{:.4f}", "Speedup (core only)": "{:,.1f}x"})

### Timing results for the three required configurations (template of the assignment)

 * 'Baseline' is **my Lab 1 code** (as submitted); 'Lab 8' is '```MosaicBuilder.build```' (same three output images)
 * The speedup target of the assignment is **at least $20\times$** for '$512\times512$' / '$32\times32$'

In [ ]:
## ********** Required configurations in the format of the assignment **********
required = [(256, 16), (512, 32), (1024, 64)]

## Rows required by the assignment
rows = comparison[[(row["size"], row["grid"]) in required for _, row in comparison.iterrows()]]

## Table with the exact columns requested
timing_template = pd.DataFrame({
    "Image size": rows["Image size"].str.replace("x", "×").to_numpy(),
    "Grid size": rows["Grid size"].str.replace("x", "×").to_numpy(),
    "Baseline time (s)": rows["Mean (s)"].to_numpy(),
    "Lab 8 time (s)": rows["lab8"].to_numpy(),
    "Speedup": rows["Speedup"].to_numpy(),
})

## Display the template
display(timing_template.style.format({"Baseline time (s)": "{:.4f}", "Lab 8 time (s)": "{:.5f}", "Speedup": "{:,.1f}x"}))

## Check the performance target on the 512 x 512 / 32 x 32 configuration
target_row = comparison[(comparison["size"] == 512) & (comparison["grid"] == 32)].iloc[0]

## Speedup of the target case
print(f"\nSpeedup at 512x512 / 32x32: {target_row['Speedup']:.1f}x  ->  target of 20x {'ACHIEVED' if target_row['Speedup'] >= 20 else 'NOT achieved'}")

## Speedup of the other views
print(f"Speedup including the metrics: {target_row['Speedup (+ metrics)']:.1f}x | core only: {target_row['Speedup (core only)']:.1f}x")

**How to read the three speedups:**
 * **Speedup** (the assignment's number) compares the _mosaic generation_ of Lab 8 with the _whole_ Lab 1 call, which also ran the metrics and the benchmark in every request - the profiler showed that this extra work, not the mosaic code, was most of the time
 * **Speedup (+ metrics)** adds MSE, PSNR and SSIM to the Lab 8 side: it is **much smaller** because the SSIM of '```scikit-image```' is used **unchanged** (to keep the metric identical) and it costs more than the whole optimized mosaic - computing it **on demand** is therefore the right design, and a faster SSIM is the natural next optimization
 * **Speedup (core only)** removes the metrics and the benchmark from Lab 1 as well: it is the gain of the **mosaic code alone** (vectorized integer reductions, cached tiles, one match, one assembly)

In [ ]:
## ********** Figures: execution time and speedup of the three required configurations **********
labels = [f"{r['Image size'].split('x')[0]} px\n{r['Grid size']} grid" for _, r in rows.iterrows()]

## Bar positions
positions = np.arange(len(labels))

## Bar width
width = 0.34

## Figure 1: execution time, baseline against Lab 8 (log axis)
fig, ax = plt.subplots(figsize = (7.2, 3.9))

## Bars of the baseline
bars_base = ax.bar(positions - width / 2, rows["Mean (s)"], width, color = COLORS["baseline"], label = "Baseline (Lab 1)")

## Bars of the optimized code
bars_opt = ax.bar(positions + width / 2, rows["lab8"], width, color = COLORS["optimized"], label = "Optimized (Lab 8)")

## Loop over both series
for bars in (bars_base, bars_opt):
    ## Loop over the bars
    for bar in bars:
        ## Label every bar with its time
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.12, format_seconds(bar.get_height()), ha = "center", fontsize = 8.5, color = INK_SECONDARY)

## Axes, legend and title
ax.set_yscale("log")

## Vertical limits
ax.set_ylim(rows["lab8"].min() / 3, rows["Mean (s)"].max() * 5)

## Tick labels
ax.set_xticks(positions, labels)

## Label of the vertical axis
ax.set_ylabel("Execution time (log scale)")

## Title of the figure
ax.set_title("Execution time of one mosaic", loc = "left")

## Legend
ax.legend(loc = "upper left")

## Hide the vertical grid
ax.grid(axis = "x", visible = False)

## Avoid overlapping elements
plt.tight_layout()

## Save the figure
plt.savefig(RESULTS / "fig_time.png")

## Display the figure
plt.show()

## Figure 2: speedup factors with the 20x target as a reference line
fig, ax = plt.subplots(figsize = (7.2, 3.6))

## Bars of the speedup
bars = ax.bar(positions, rows["Speedup"], 0.5, color = COLORS["optimized"])

## Loop over the bars
for bar, value in zip(bars, rows["Speedup"]):
    ## Label every bar with its speedup
    ax.text(bar.get_x() + bar.get_width() / 2, value * 1.03, f"{value:,.0f}x", ha = "center", fontsize = 9.5, color = INK_SECONDARY)

## Target line, axes and title
ax.axhline(20, color = INK_SECONDARY, linestyle = "--", linewidth = 1)

## Mark the target
ax.text(len(labels) - 0.5, 21, "target: 20x", ha = "right", fontsize = 9, color = INK_SECONDARY)

## Tick labels
ax.set_xticks(positions, labels)

## Label of the vertical axis
ax.set_ylabel("Speedup over Lab 1 (x)")

## Vertical limits
ax.set_ylim(0, rows["Speedup"].max() * 1.18)

## Title of the figure
ax.set_title("Speedup of Lab 8 over the Lab 1 baseline", loc = "left")

## Hide the vertical grid
ax.grid(axis = "x", visible = False)

## Avoid overlapping elements
plt.tight_layout()

## Save the figure
plt.savefig(RESULTS / "fig_speedup.png")

## Display the figure
plt.show()

In [ ]:
## ********** Figure: scaling with image size, one panel per grid size **********
fig, axes = plt.subplots(1, 3, figsize = (11, 3.6), sharey = True)

## One panel per grid size, two lines (baseline and optimized) over the image sizes
for axis, grid in zip(axes, GRID_SIZES):
    ## Rows of this grid size
    part = comparison[comparison["grid"] == grid]

    ## Line of the baseline
    axis.plot(part["size"], part["Mean (s)"], marker = "o", markersize = 6, linewidth = 2, color = COLORS["baseline"], label = "Baseline (Lab 1)")

    ## Line of the optimized code
    axis.plot(part["size"], part["lab8"], marker = "o", markersize = 6, linewidth = 2, color = COLORS["optimized"], label = "Optimized (Lab 8)")

    ## Axes of the panel
    axis.set_xscale("log", base = 2)

    ## Logarithmic vertical axis
    axis.set_yscale("log")

    ## Tick labels
    axis.set_xticks(IMAGE_SIZES, [str(size) for size in IMAGE_SIZES])

    ## Label of the horizontal axis
    axis.set_xlabel("Image side (px)")

    ## Title of the panel
    axis.set_title(f"{grid} x {grid} grid", loc = "left")

## Shared y label and legend
axes[0].set_ylabel("Execution time (s, log scale)")

## Legend
axes[0].legend(loc = "center left", bbox_to_anchor = (0.0, 0.5))

## Avoid overlapping elements
plt.tight_layout()

## Save the figure
plt.savefig(RESULTS / "fig_scaling.png")

## Display the figure
plt.show()

**Scaling behavior:**
 * The Lab 1 time depends almost only on the **image side**, not on the grid: the cost of its two big items (SSIM and the benchmark) is proportional to the number of **pixels** - it grows roughly $4\times$ for every doubling of the side
 * The optimized time also grows with the **number of pixels** (the resize, the reduction and the final assembly all touch every pixel once), but with a **much smaller constant**
 * The grid size matters little in both versions: the number of cells ($256$ to $4{,}096$) is small compared with the number of pixels, and every per-cell step is vectorized

### Correctness check: Lab 1 against the optimized code

 * **All four** of my sample photos (original size, **not** pre-resized), **all three** tile families and the **three required configurations** are compared: $4\times3\times3=36$ cases
 * For every case, three numbers: whether the **mosaic is identical pixel by pixel**, the **MSE** between both mosaics, and the **percentage of cells that received the same tile**
 * The tile indices of Lab 1 are recomputed with its own '```classify_cells```', because Lab 1 does not return them

In [ ]:
## ********** Correctness: 4 photos x 3 tile families x 3 configurations **********
sample_paths = sorted((PROJECT / "sample_images").glob("*.jpeg"))

## Rows of the table
correctness_rows = []

## Loop over the sample images
for path in sample_paths:
    ## Read the image
    raw = np.asarray(Image.open(path).convert("RGB"))

    ## Loop over the tile families
    for tile_set in ("Solid", "Diagonal", "Checkerboard"):
        ## Loop over the cases
        for size, grid in [(256, 16), (512, 32), (1024, 64)]:

            ## Lab 1 mosaic and the tile index of every cell (recomputed with the Lab 1 functions)
            with contextlib.redirect_stdout(io.StringIO()):
                ## Lab 1 pipeline
                lab1_prepared, lab1_segmented, lab1_mosaic = mosaic_pipeline.create_mosaic(raw, grid, tile_set, size)

            ## Lab 1 cell means
            lab1_means = grid_analysis.cell_means_vectorized(lab1_prepared, grid)

            ## Lab 1 tile indices
            lab1_indices = mosaic_mapping.classify_cells(lab1_means, tile_library.create_tile_set(size // grid, tile_set))

            ## Optimized mosaic of the same photo
            result = MosaicBuilder(TileManager(tile_set = tile_set), grid_size = grid, image_size = size).build(raw)

            ## The three numbers of the check
            correctness_rows.append({
                "Photo": path.stem[-1], "Tiles": tile_set, "Config": f"{size}/{grid}",
                "Identical mosaic": bool(np.array_equal(lab1_mosaic, result.mosaic)),
                "Identical segmentation": bool(np.array_equal(lab1_segmented, result.segmented)),
                "MSE (Lab 1 vs. Lab 8)": lab8_metrics.mean_squared_error(lab1_mosaic, result.mosaic),
                "Matching tiles (%)": 100 * lab8_metrics.tile_agreement(lab1_indices, result.tile_indices),
            })

## Summary of the 36 cases
correctness_table = pd.DataFrame(correctness_rows)

## Number of cases
print(f"Cases checked                     : {len(correctness_table)}")

## Identical mosaics
print(f"Identical mosaics                 : {int(correctness_table['Identical mosaic'].sum())} / {len(correctness_table)}")

## Identical segmentations
print(f"Identical segmentations           : {int(correctness_table['Identical segmentation'].sum())} / {len(correctness_table)}")

## Largest MSE
print(f"Largest MSE between both mosaics  : {correctness_table['MSE (Lab 1 vs. Lab 8)'].max():.6f}")

## Lowest agreement
print(f"Lowest percentage of matching tiles: {correctness_table['Matching tiles (%)'].min():.2f} %")

## Display the first rows
correctness_table.head(9)

### Algorithmic improvements: a real tile library and the k-d tree

 * Lab 1 matched every cell to **four** tiles, so a nearest-neighbor _structure_ cannot show its value; the package also loads a **library of $216$ image tiles** from disk (the folder '```tiles```', made by '```generate_tiles.py```': $6^3$ colors, six textures)
 * The tiles are **read once** at startup and **resized once per cell size** (area averaging when shrinking), then served from the cache
 * Two **matching features** are compared: the **mean color** of a cell ('$3$ numbers', Lab 1 behavior) and the **quadrant colors** (the mean of each of its $2\times2$ sub-blocks, '$12$ numbers') - the second sees _structure inside_ the cell
 * Two **search strategies**: '```cKDTree```' (tree built once per cell size) and '```brute```' (distances to every tile through one matrix product)

In [ ]:
## ********** File-based tile library: matching feature and search strategy at 512 x 512 / 64 x 64 **********
file_manager = TileManager(tile_directory = PROJECT / "tiles")

## Image of 512 pixels
image_512 = test_images[512]

## Every combination of feature and search strategy
library_rows = []

## Results of every combination
library_mosaics = {}

## Loop over the matching modes
for mode in ("mean", "quadrant"):
    ## Loop over the search strategies
    for algorithm in ("kdtree", "brute"):
        ## Builder of this combination
        builder = MosaicBuilder(file_manager, grid_size = 64, image_size = 512, match_mode = mode, match_algorithm = algorithm)

        ## Time of the whole build and of the matching stage alone
        summary = time_callable(lambda: builder.build(image_512, segmentation = False), repeats = OPTIMIZED_RUNS, warmup = 2)

        ## Build the mosaic
        result = builder.build(image_512, segmentation = False)

        ## Quality of the mosaic against the prepared image
        quality = builder.compute_similarity(result.prepared, result.mosaic)

        ## Keep the result
        library_mosaics[(mode, algorithm)] = result

        ## Store the row of the table
        library_rows.append({"Feature": mode, "Search": algorithm, "Build (ms)": summary["mean"] * 1e3, "Match stage (ms)": result.timings["match"] * 1e3,
                             "MSE": quality["MSE"], "PSNR (dB)": quality["PSNR"], "SSIM": quality["SSIM"]})

## Share of cells for which the k-d tree and brute force choose the same tile (differences can only be exact ties)
for mode in ("mean", "quadrant"):
    ## Agreement of both strategies
    agreement = 100 * lab8_metrics.tile_agreement(library_mosaics[(mode, "kdtree")].tile_indices, library_mosaics[(mode, "brute")].tile_indices)

    ## Report the agreement
    print(f"k-d tree vs. brute force, '{mode}' feature: {agreement:.2f} % of the cells get the same tile")

## Table of the four combinations
library_table = pd.DataFrame(library_rows)

## Display the table
library_table

In [ ]:
## ********** Figure: the same photo with the Lab 1 tiles and with the tile library **********
lab1_result = MosaicBuilder(TileManager(tile_set = "Diagonal"), grid_size = 64, image_size = 512).build(image_512, segmentation = False)

## Panels of the figure
panels = [
    ("Input (512 x 512)", library_mosaics[("mean", "kdtree")].prepared),
    ("Lab 1 tiles ('Diagonal', 4 tiles)", lab1_result.mosaic),
    ("216 library tiles, mean color", library_mosaics[("mean", "kdtree")].mosaic),
    ("216 library tiles, quadrant colors", library_mosaics[("quadrant", "kdtree")].mosaic),
]

## Four panels side by side, 64 x 64 cells
fig, axes = plt.subplots(1, 4, figsize = (13, 3.7))

## Loop over the panels
for axis, (title, image) in zip(axes, panels):
    ## Show the image
    axis.imshow(image)

    ## Title of the panel
    axis.set_title(title, fontsize = 9)

    ## Hide the axes
    axis.axis("off")

## Avoid overlapping elements
plt.tight_layout()

## Save the figure
plt.savefig(RESULTS / "fig_mosaic_examples.png")

## Display the figure
plt.show()

In [ ]:
## ********** Search strategies against the number of tiles (4096 cells) **********
rng = np.random.default_rng(0)

## Queries of 3 features
queries_3d = rng.uniform(0, 255, size = (4096, 3))

## Queries of 12 features
queries_12d = rng.uniform(0, 255, size = (4096, 12))

## This function finds the nearest center of every query with one matrix product
def brute_force(queries: np.ndarray, centers: np.ndarray) -> np.ndarray:
    """ Index of the nearest center (squared Euclidean distance) of every query """

    ## Expand |q - c|^2 = |q|^2 - 2 q.c + |c|^2 and take the arg-min per query
    squared = (queries ** 2).sum(axis = 1)[:, None] - 2.0 * queries @ centers.T + (centers ** 2).sum(axis = 1)[None, :]

    ## Index of the nearest center
    return squared.argmin(axis = 1)


## Timings for several library sizes and for 3 and 12 features per tile
scaling_rows = []

## Loop over the feature sizes
for features, queries in ((3, queries_3d), (12, queries_12d)):
    ## Loop over the numbers of tiles
    for n_tiles in (4, 64, 216, 1000, 5000, 20000):
        ## Random centers
        centers = rng.uniform(0, 255, size = (n_tiles, features))

        ## Tree built once (cached by the TileManager), then queried; brute force has no build step
        build = time_callable(lambda: cKDTree(centers), repeats = 5, warmup = 1)["mean"]

        ## Build the tree
        tree = cKDTree(centers)

        ## Time the queries of the tree
        query = time_callable(lambda: tree.query(queries, k = 1), repeats = 10, warmup = 1)["mean"]

        ## Time the brute-force search
        brute = time_callable(lambda: brute_force(queries, centers), repeats = 10, warmup = 1)["mean"]

        ## Both strategies must agree
        agree = bool(np.array_equal(tree.query(queries, k = 1)[1], brute_force(queries, centers)))

        ## Store the row of the table
        scaling_rows.append({"Features": features, "Tiles": n_tiles, "Tree build (ms)": build * 1e3, "Tree query (ms)": query * 1e3, "Brute force (ms)": brute * 1e3, "Same answer": agree})

## Show the table
kdtree_table = pd.DataFrame(scaling_rows)

## Display the table
kdtree_table

**What the library experiment shows:**
 * The **quadrant** feature gives a lower error than the mean color (more of the cell's structure is matched), at a small cost in the grid stage - a **better algorithm**, not only a faster one
 * With _few_ tiles, the matrix-product search is **faster** than the tree - the tree pays off when the library grows (look at the rows with hundreds or thousands of tiles) and it is built **once** and cached
 * The two strategies return the **same tiles** except when a cell is _exactly_ as far from two tiles (two dark tiles with the same clipped mean color): then either answer is a nearest tile, so the choice is purely about speed
 * With the **four** tiles of Lab 1 the matrix product is faster than the tree, which is why the default strategy is '```auto```' (tree from $128$ tiles on for $3$ features, from $4{,}096$ tiles on for $12$ features)

### Efficient resizing: the interpolation filter

 * The input is resized **once** ('```prepare_image```'); the filter can be chosen
 * Here a **full-size camera photo** ('$1570\times1600$') is reduced to '$512\times512$' with every filter: the time of the resize, the **MSE** of the prepared image against the '```lanczos```' one (the Lab 1 filter) and the **percentage of cells that get the same tile** (216-tile library)

In [ ]:
## ********** Resampling filter study on a full-size photo **********
camera_photo = Image.open(SAMPLE).convert("RGB")

## Reference builder with Lanczos
reference_builder = MosaicBuilder(file_manager, grid_size = 32, image_size = 512, resample = "lanczos")

## Reference result
reference_run = reference_builder.build(camera_photo, segmentation = False)

## One row per filter
resample_rows = []

## Loop over the filters
for name in ("lanczos", "bicubic", "bilinear", "box"):
    ## Builder of this filter
    builder = MosaicBuilder(file_manager, grid_size = 32, image_size = 512, resample = name)

    ## Time of the preparation alone and quality against the Lanczos result
    prepare_time = time_callable(lambda: image_processor.prepare_image(camera_photo, 512, name), repeats = 20, warmup = 2)["mean"]

    ## Result of this filter
    run = builder.build(camera_photo, segmentation = False)

    ## Store the row of the table
    resample_rows.append({
        "Filter": name, "Resize (ms)": prepare_time * 1e3,
        "MSE of the prepared image vs. lanczos": lab8_metrics.mean_squared_error(reference_run.prepared, run.prepared),
        "Cells with the same tile (%)": 100 * lab8_metrics.tile_agreement(reference_run.tile_indices, run.tile_indices),
    })

## Show the table
resample_table = pd.DataFrame(resample_rows)

## Display the table
resample_table

### Caching: what loading the tiles on every request would cost

 * Lab 1 had no file I/O, but a typical first version of a **tile-folder** mosaic reloads the files on every request - the situation that the assignment calls _repeated file loading_
 * Compare **(a)** a '```TileManager```' created _inside every request_ (files read and resized each time), **(b)** one manager created at startup, **cold** (first request builds the cache) and **warm** (cache hits)

In [ ]:
## ********** Cost of reloading the tiles on every request (216 tiles, 512 x 512 / 32 x 32) **********
image = test_images[512]

## (a) A new manager for every request: files are read and resized every time
def reload_every_request():
    """ Build a manager and a builder inside the request, as a naive implementation would """

    ## All the file I/O happens here, on every call
    manager = TileManager(tile_directory = PROJECT / "tiles")

    ## Builder that reloads the tiles on every request
    return MosaicBuilder(manager, grid_size = 32, image_size = 512).create_mosaic(image)


## (b) One manager at startup: the first request is cold, the next ones are warm
startup = time.perf_counter()

## Manager shared between requests
shared_manager = TileManager(tile_directory = PROJECT / "tiles")

## Start-up time
startup_seconds = time.perf_counter() - startup

## Builder that uses the shared manager
shared_builder = MosaicBuilder(shared_manager, grid_size = 32, image_size = 512)

## Time of the first (cold) request and of the warm requests
start = time.perf_counter()

## Fill the cache
shared_builder.create_mosaic(image)

## Cold time
cold_seconds = time.perf_counter() - start

## Warm time
warm = time_callable(lambda: shared_builder.create_mosaic(image), repeats = OPTIMIZED_RUNS, warmup = 1)["mean"]

## Time with reloading
reload = time_callable(reload_every_request, repeats = 10, warmup = 1)["mean"]

## Summary table
cache_table = pd.DataFrame({
    "Scenario": ["(a) reload tiles on every request", "(b) one manager: startup (once)", "(b) first request (cold cache)", "(b) next requests (warm cache)"],
    "Time (ms)": [reload * 1e3, startup_seconds * 1e3, cold_seconds * 1e3, warm * 1e3],
}).set_index("Scenario")

## Report the cache counters
print(f"Cache hits / misses of the shared manager: {shared_manager.cache_hits} / {shared_manager.cache_misses}")

## Report the speedup of the cache
print(f"Warm request is {reload / warm:.0f}x faster than reloading the tiles on every request")

## Display the table
cache_table

## Part 3: Refactoring into the '```mosaic_generator```' package

The optimized code lives in a package with **one responsibility per module** (the structure requested in the assignment):

 * '```config.py```' - every constant (grid sizes, tile families, limits, defaults)
 * '```utils.py```' - exceptions, timing helpers and the validation of grids, sizes and options
 * '```image_processor.py```' - loading, cropping/resizing, grid division, cell means, segmentation and mosaic assembly
 * '```tile_manager.py```' - **'```TileManager```'**: loads the tiles once, caches resized tiles, features and k-d trees, answers nearest-tile queries
 * '```mosaic_builder.py```' - **'```MosaicBuilder```'**: the reconstruction algorithm and the per-stage timings
 * '```metrics.py```' - MSE, PSNR, SSIM and tile agreement
 * '```app.py```' (outside the package) - the web interface, which only calls the package

In [ ]:
## ********** Package structure **********
for path in sorted((PROJECT / "mosaic_generator").glob("*.py")):

    ## Number of lines of every module
    print(f"mosaic_generator/{path.name:<20} {len(path.read_text(encoding = 'utf-8').splitlines()):>4} lines")

In [ ]:
## ********** The API of the assignment, as written there **********
tile_manager = TileManager(tile_directory = PROJECT / "tiles")

## Create the builder
builder = MosaicBuilder(tile_manager, grid_size = (32, 32))

## Generate a mosaic and measure its similarity with the input
input_image = np.asarray(Image.open(SAMPLE).convert("RGB"))

## Create the mosaic
mosaic = builder.create_mosaic(input_image)

## Similarity metrics
similarity = builder.compute_similarity(input_image, mosaic)

## Show the result and the per-stage timings of the build
print("Mosaic shape:", mosaic.shape)

## Report the similarity
print("Similarity  :", {key: round(value, 4) for key, value in similarity.items()})

## Report the stage times
print("Stage times :", {key: format_seconds(value) for key, value in builder.last_timings.items()})

In [ ]:
## ********** Unit tests (pytest) **********
completed = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "--no-header", "--color=no", "-p", "no:cacheprovider"], cwd = PROJECT, capture_output = True, text = True)

## Last lines of the pytest report
print("\n".join(completed.stdout.strip().splitlines()[-4:]))

In [ ]:
## ********** Save every number of this Notebook for the performance report **********
## This function turns a DataFrame into plain Python records
def records(frame: pd.DataFrame) -> list[dict]:
    """ DataFrame rows as dictionaries with plain Python numbers """

    ## JSON needs plain types
    return json.loads(frame.reset_index().to_json(orient = "records"))


## Everything the report needs, in one dictionary
results = {
    "machine": machine,
    "settings": {"tile_set": TILE_SET, "baseline_runs": BASELINE_RUNS, "optimized_runs": OPTIMIZED_RUNS, "sample": SAMPLE.name},
    "baseline": records(baseline_table.drop(columns = ["size", "grid"])),
    "stages": records(stage_table),
    "bottlenecks": records(bottleneck_table),
    "ladder": records(ladder_table),
    "comparison": records(comparison),
    "required": records(timing_template),
    "target": {"speedup": float(target_row["Speedup"]), "speedup_with_metrics": float(target_row["Speedup (+ metrics)"]),
               "speedup_core_only": float(target_row["Speedup (core only)"]), "achieved": bool(target_row["Speedup"] >= 20)},
    "correctness": {"cases": len(correctness_table), "identical_mosaics": int(correctness_table["Identical mosaic"].sum()),
                    "identical_segmentations": int(correctness_table["Identical segmentation"].sum()),
                    "max_mse": float(correctness_table["MSE (Lab 1 vs. Lab 8)"].max()), "min_matching_tiles_pct": float(correctness_table["Matching tiles (%)"].min())},
    "library": records(library_table),
    "kdtree_scaling": records(kdtree_table),
    "resampling": records(resample_table),
    "cache": records(cache_table),
    "unit_tests": "\n".join(completed.stdout.strip().splitlines()[-1:]),
}

## Write the file next to the figures
(RESULTS / "benchmark_results.json").write_text(json.dumps(results, indent = 2), encoding = "utf-8")

## Report the saved file
print("Saved:", RESULTS / "benchmark_results.json")

## Conclusions

 * **Profile first:** the two biggest costs of Lab 1 were **not** in the mosaic code at all - they were the SSIM and a loop benchmark executed in every request; only the profiler could show that
 * **Vectorize the rest:** contiguous integer reductions, one broadcast, one fancy-indexing gather and one transpose build the whole mosaic with **no loop over cells or pixels**
 * **Cache what never changes:** tiles, their features and the k-d tree are computed once per cell size; the tile files are read once at startup
 * **Verify:** the optimized mosaics are **identical** to the Lab 1 ones in all $36$ checked cases, so the speedup costs no quality
 * **Be honest about the scale:** the speedup depends on _what is counted_ - removing work that was never part of the mosaic gives most of the gain, while the faster mosaic code alone gives a smaller (still large) factor; the three columns reported above show each view
 * **Maintainable:** the same code is now a package with docstrings, input validation, unit tests and a web interface that only calls its API

# External references

 * Python Software Foundation. (n.d.). The Python profilers. Python documentation. Retrieved from: https://docs.python.org/3/library/profile.html

 * Kern, R. (n.d.). line_profiler: Line-by-line profiler. Retrieved from: https://github.com/pyutils/line_profiler

 * Harris, C. R., Millman, K. J., van der Walt, S. J., et al. (2020). Array programming with NumPy. Nature, 585(7825), 357-362. https://doi.org/10.1038/s41586-020-2649-2

 * SciPy Developers. (n.d.). scipy.spatial.cKDTree. SciPy manual. Retrieved from: https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.cKDTree.html

 * Wang, Z., Bovik, A. C., Sheikh, H. R., & Simoncelli, E. P. (2004). Image quality assessment: from error visibility to structural similarity. IEEE Transactions on Image Processing, 13(4), 600-612. https://doi.org/10.1109/TIP.2003.819861

 * van der Walt, S., Schönberger, J. L., Nunez-Iglesias, J., et al. (2014). scikit-image: image processing in Python. PeerJ, 2, e453. https://doi.org/10.7717/peerj.453

 * Bentley, J. L. (1975). Multidimensional binary search trees used for associative searching. Communications of the ACM, 18(9), 509-517. https://doi.org/10.1145/361002.361007